# Convertir rutas de ruta directa a geojson

In [1]:
import pandas as pd
import json

import geopandas as gpd

from pypolyline.cutil import encode_coordinates, decode_polyline



In [5]:
path_base = "/make_gtfs/data/tampico/"
estado = "tampico"
ciudad = "tampico"

### Leer archivo

In [7]:
# Leer archivo JSON desde disco
with open(f'./data/raw/{ciudad}/rutas-ruta_directa.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# Si el JSON tiene una lista de rutas como en tu ejemplo:
df = pd.json_normalize(data['routes'])
df.drop(columns=["data.route.id", "data.route.slug",  "data.route.type", 	"data.route.color", 	"data.route.entity.id", 	"data.route.entity.__typename", 	"data.route.trips", "data.route.__typename"], inplace=True)
# Mostrar las primeras filas
df.head()

,data.route.encodedLine,data.route.shortName,data.route.longName
0,u~qfCrcgtQxRnIf@gEoQ}HaIqDaAvDsCyAcRyHiYkM_WyK...,1,Mirador - Aviación por Boulevard
1,}sigCr~ltQkF}CYr@]bGHf@hM`Ip@?Re@ZaGAY_GuDtIgF...,2,Tampico - Bosque Central de Abastos - Casa Blanca
2,ssigCf~ltQ}F}CKv@_@tGZr@nMjHr@]VoHkGiDrIiF}C}G...,3,Madero - Revolución Verde - Central de Abastos
3,ajggCz`ptQgB?rEoa@xObDrLzBjE~A|@?K_AfFiBU}AjBm...,4,Colosio - Águila Madero - Golfo
4,afggCbjhtQdDoAbCnKeDtA~@Cr@dDaKvCiIrCm^tK}CjAO...,5,Madero - Blanco Palmas - Albañiles


### convertir encoded line a shape

In [8]:
df["decoded_line"] = df["data.route.encodedLine"].map(lambda x: decode_polyline(x.encode('utf-8'), 5))
df.head()

,data.route.encodedLine,data.route.shortName,data.route.longName,decoded_line
0,u~qfCrcgtQxRnIf@gEoQ}HaIqDaAvDsCyAcRyHiYkM_WyK...,1,Mirador - Aviación por Boulevard,"[[-97.85418, 22.21563], [-97.85586, 22.21246],..."
1,}sigCr~ltQkF}CYr@]bGHf@hM`Ip@?Re@ZaGAY_GuDtIgF...,2,Tampico - Bosque Central de Abastos - Casa Blanca,"[[-97.8841, 22.33679], [-97.88331, 22.33797], ..."
2,ssigCf~ltQ}F}CKv@_@tGZr@nMjHr@]VoHkGiDrIiF}C}G...,3,Madero - Revolución Verde - Central de Abastos,"[[-97.88404, 22.33674], [-97.88325, 22.33801],..."
3,ajggCz`ptQgB?rEoa@xObDrLzBjE~A|@?K_AfFiBU}AjBm...,4,Colosio - Águila Madero - Golfo,"[[-97.89982, 22.32497], [-97.89982, 22.32549],..."
4,afggCbjhtQdDoAbCnKeDtA~@Cr@dDaKvCiIrCm^tK}CjAO...,5,Madero - Blanco Palmas - Albañiles,"[[-97.86034, 22.32433], [-97.85994, 22.3235], ..."


## Convertir a lionestring y luego a geopandas

In [9]:
import geopandas as gpd
from shapely.geometry import LineString

# Suponiendo que tu DataFrame se llama df
# decoded_line tiene listas de [lon, lat] o [lat, lon], verifica orden

# GeoJSON y Google Polylines suelen usar [lat, lon], pero Shapely espera (x, y) = (lon, lat)
# Si tu decoded_line es [[lon, lat], ...] perfecto
# Si es [[lat, lon], ...] debes invertir coordenadas

# Si las coordenadas están en orden [lon, lat] (como en tu ejemplo), solo:

df['geometry'] = df['decoded_line'].apply(lambda coords: LineString(coords))

# Crear GeoDataFrame con el sistema de referencia WGS84 (EPSG:4326)
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

# Mostrar resultado

### Calcula lenght
luego se usa para filtrar rutas muy cortas o con nans

In [10]:
# Primero, reproyectar a CRS métrico, ejemplo UTM zona 14N (ajusta según ubicación)
gdf_utm = gdf.to_crs(epsg=32614)

# Calcular longitud en metros
gdf_utm['length_m'] = gdf_utm.geometry.length

# Si quieres mantenerlo en el gdf original, puedes asignar:
gdf['length_m'] = gdf_utm['length_m']
gdf.sort_values("length_m", inplace=True)
gdf[['data.route.shortName', 'length_m']]

gdf.dropna(subset="length_m", inplace=True)

/Users/danielbustillos/miniconda3/envs/analisis-general/lib/python3.10/site-packages/shapely/measurement.py:182: RuntimeWarning: invalid value encountered in length
  return lib.length(geometry, **kwargs)


In [11]:
gdf.head()

,data.route.encodedLine,data.route.shortName,data.route.longName,decoded_line,geometry,length_m
71,avpfC`ietQaCvA]}AkCaEiFqJIJnJtP`@vAcGzDmCbIsQ`...,74,Isleta Pérez,"[[-97.84481, 22.20913], [-97.84525, 22.20978],...","LINESTRING (-97.84481 22.20913, -97.84525 22.2...",3975.775449
73,wdtfC`pjtQFo@\UbBsBh]aIxCq@`Bo@GU|Zo}@|HfD~CuI...,76,Cascajal,"[[-97.87153, 22.22684], [-97.87129, 22.2268], ...","LINESTRING (-97.87153 22.22684, -97.87129 22.2...",5008.043778
78,{atfChgdtQqAtErCc@rLlFhAoDzCtAhAmD`p@tX~AzFh@b...,81,Golfo,"[[-97.83941, 22.22638], [-97.84048, 22.22679],...","LINESTRING (-97.83941 22.22638, -97.84048 22.2...",7057.923626
88,yjxgC~fttQWfE~PnAVaEzv@pFfDr@pKrc@`ArBpN|L\zBa...,89A,Altamira - Tampiquito por Soriana,"[[-97.92128, 22.41213], [-97.92228, 22.41225],...","LINESTRING (-97.92128 22.41213, -97.92228 22.4...",7208.457913
64,ixxfCv{gtQpWgBlFoM}ISyEgBiLr@jBeE??lIk@vHwQgCs...,67,Central Camionera - Madero,"[[-97.85804, 22.25045], [-97.85752, 22.24652],...","LINESTRING (-97.85804 22.25045, -97.85752 22.2...",7498.392531


### Ordenar tabla en formato routes para gtfs

In [12]:
gdf.drop(columns=["data.route.encodedLine", "decoded_line", "length_m"], inplace=True)
gdf.head()

,data.route.shortName,data.route.longName,geometry
71,74,Isleta Pérez,"LINESTRING (-97.84481 22.20913, -97.84525 22.2..."
73,76,Cascajal,"LINESTRING (-97.87153 22.22684, -97.87129 22.2..."
78,81,Golfo,"LINESTRING (-97.83941 22.22638, -97.84048 22.2..."
88,89A,Altamira - Tampiquito por Soriana,"LINESTRING (-97.92128 22.41213, -97.92228 22.4..."
64,67,Central Camionera - Madero,"LINESTRING (-97.85804 22.25045, -97.85752 22.2..."


In [15]:
gdf["shape_id"] = 	"shape_" +  gdf["data.route.shortName"]	
gdf.head()

,data.route.shortName,data.route.longName,geometry,shape_id
71,74,Isleta Pérez,"LINESTRING (-97.84481 22.20913, -97.84525 22.2...",shape_74
73,76,Cascajal,"LINESTRING (-97.87153 22.22684, -97.87129 22.2...",shape_76
78,81,Golfo,"LINESTRING (-97.83941 22.22638, -97.84048 22.2...",shape_81
88,89A,Altamira - Tampiquito por Soriana,"LINESTRING (-97.92128 22.41213, -97.92228 22.4...",shape_89A
64,67,Central Camionera - Madero,"LINESTRING (-97.85804 22.25045, -97.85752 22.2...",shape_67


In [16]:
gdf.to_file(f"./data/proc/{estado}/rutas_procesadas.geojson", driver="GeoJSON")

In [17]:
gdf.head()

,data.route.shortName,data.route.longName,geometry,shape_id
71,74,Isleta Pérez,"LINESTRING (-97.84481 22.20913, -97.84525 22.2...",shape_74
73,76,Cascajal,"LINESTRING (-97.87153 22.22684, -97.87129 22.2...",shape_76
78,81,Golfo,"LINESTRING (-97.83941 22.22638, -97.84048 22.2...",shape_81
88,89A,Altamira - Tampiquito por Soriana,"LINESTRING (-97.92128 22.41213, -97.92228 22.4...",shape_89A
64,67,Central Camionera - Madero,"LINESTRING (-97.85804 22.25045, -97.85752 22.2...",shape_67


In [18]:
gdf.explore(column="data.route.shortName", categorical=True,  )